In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
import torchvision.models as models
import os
 
from olimp.dataset.olimp import olimp
from olimp.dataset import read_img_path

from olimp.dataset.olimp import olimp
from olimp.dataset import read_img_path

/home/user/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#  ЯЧЕЙКА 2: Конфигурация
# ──────────────────────────────────────────────────────────

PATCH_SIZE        = 256
SIGMA             = 10 / 255.0
TRAIN_RATIO       = 0.85

PATCHES_PER_EPOCH = 50_000
VAL_PATCHES       = 5_000

BATCH_SIZE        = 32
NUM_WORKERS       = 4
EPOCHS            = 300
PATIENCE          = 30
LR                = 1e-3

CACHE_IMAGES      = True

CHECKPOINT_PATH   = "checkpoint(10).pt"
BEST_MODEL_PATH   = "best_model(10).pt"
ALL_MODELS_DIR    = "all_models(10)"

CHECKPOINT_EVERY  = 5

os.makedirs(ALL_MODELS_DIR, exist_ok=True)

In [3]:
CATEGORIES = [
    'abstracts and textures', 'abstracts and textures/abstract art',
    'abstracts and textures/backgrounds and patterns',
    'abstracts and textures/colorful abstracts',
    'abstracts and textures/geometric shapes',
    'abstracts and textures/neon abstracts', 'abstracts and textures/textures',
    'animals', 'animals/birds', 'animals/farm animals',
    'animals/insects and spiders', 'animals/marine life', 'animals/pets',
    'animals/wild animals', 'art and culture',
    'art and culture/cartoon and comics',
    'art and culture/crafts and handicrafts',
    'art and culture/dance and theater performances',
    'art and culture/music concerts and instruments',
    'art and culture/painting and frescoes',
    'art and culture/sculpture and bas-reliefs', 'food and drinks',
    'food and drinks/desserts and bakery', 'food and drinks/dishes',
    'food and drinks/drinks',
    'food and drinks/food products on store shelves',
    'food and drinks/fruits and vegetables', 'food and drinks/street food',
    'interiors', 'interiors/gyms and pools', 'interiors/living spaces',
    'interiors/museums and galleries', 'interiors/offices',
    'interiors/restaurants and cafes',
    'interiors/shopping centers and stores', 'nature', 'nature/beaches',
    'nature/deserts', 'nature/fields and meadows', 'nature/forest',
    'nature/mountains', 'nature/water bodies', 'objects and items',
    'objects and items/books and stationery',
    'objects and items/clothing and accessories',
    'objects and items/electronics and gadgets',
    'objects and items/furniture and decor',
    'objects and items/tools and equipment',
    'objects and items/toys and games', 'portraits and people',
    'portraits and people/athletes and dancers',
    'portraits and people/crowds and demonstrations',
    'portraits and people/group photos',
    'portraits and people/individual portraits',
    'portraits and people/models on runway',
    'portraits and people/workers in their workplaces',
    'sports and active leisure',
    'sports and active leisure/cycling and rollerblading',
    'sports and active leisure/extreme sports',
    'sports and active leisure/individual sports',
    'sports and active leisure/martial arts',
    'sports and active leisure/team sports',
    'sports and active leisure/tourism and hikes', 'text and pictogram',
    'text and pictogram/billboard text', 'text and pictogram/blueprints',
    'text and pictogram/caricatures and pencil drawing',
    'text and pictogram/text documents', 'text and pictogram/traffic signs',
    'urban scenes', 'urban scenes/architecture',
    'urban scenes/city at night', 'urban scenes/graffiti and street art',
    'urban scenes/parks and squares', 'urban scenes/streets and avenues',
    'urban scenes/transport',
]

all_paths = []
for cat in CATEGORIES:
    try:
        ds    = olimp(categories={cat})
        paths = ds[cat]
        all_paths.extend(paths)
    except Exception as e:
        print(f"  {cat}: пропущено ({e})")

rng = np.random.default_rng(42)
rng.shuffle(all_paths)
all_paths = all_paths[:1000]
print(f"\nИтого загружено: {len(all_paths)} изображений")

n_train     = int(TRAIN_RATIO * len(all_paths))
train_paths = all_paths[:n_train]
val_paths   = all_paths[n_train:]
print(f"Train: {len(train_paths)} изображений | Val: {len(val_paths)} изображений")

/home/user/.local/lib/python3.10/site-packages/rich/live.py:231: UserWarning: install "ipywidgets" for Jupyter 
support
  warnings.warn('install "ipywidgets" for Jupyter support')


Итого загружено: 1000 изображений
Train: 850 изображений | Val: 150 изображений


In [4]:
# ЯЧЕЙКА 4: Генерация конфигураций масок K

_cfg_rng = np.random.default_rng(0)

stripe_h_configs = [
    {"type": "h", "width": float(w)}
    for w in _cfg_rng.uniform(4, 128, 1000)
]

sin_configs = [
    {
        "type": "s",
        "freq":  float(f),
        "angle": float(a),
        "phase": float(p),
    }
    for f, a, p in zip(
        _cfg_rng.uniform(1 / 120, 1 / 8, 1000),
        _cfg_rng.uniform(0, 180, 1000),
        _cfg_rng.uniform(0, 2 * np.pi, 1000),
    )
]

ALL_CONFIGS = stripe_h_configs + sin_configs
print(f"Всего конфигураций K: {len(ALL_CONFIGS)}  "
      f"(горизонтальных: {len(stripe_h_configs)}, "
      f"синусоидальных: {len(sin_configs)})")


def make_mask(h: int, w: int, config: dict) -> np.ndarray:
    """
    Генерирует маску K ∈ [0,1].
    Используется только при создании измерения P — сети не передаётся.
    """
    if config["type"] == "h":
        sw = max(1, int(config["width"]))
        y  = np.arange(h)
        K  = ((y // sw) % 2).astype(np.float32)
        return K[:, None] * np.ones((1, w), dtype=np.float32)

    freq  = config["freq"]
    angle = np.radians(config["angle"])
    phase = config["phase"]
    a = 2 * np.pi * freq * np.cos(angle)
    b = 2 * np.pi * freq * np.sin(angle)
    xs = np.arange(w, dtype=np.float32)[None, :]
    ys = np.arange(h, dtype=np.float32)[:, None]
    return ((np.sin(a * xs + b * ys + phase) + 1) / 2).astype(np.float32)

Всего конфигураций K: 2000  (горизонтальных: 1000, синусоидальных: 1000)


In [5]:
# ЯЧЕЙКА 5: Вспомогательные функции

def load_rg(img_path):
    img = read_img_path(img_path)
    img = np.array(img, dtype=np.float32) / 255.0
    img = np.transpose(img, (1, 2, 0))
    img = cv2.resize(img, (512, 512))
    if img.ndim == 2:
        img = np.stack([img, img, img], axis=-1)
    elif img.shape[2] == 1:
        img = np.concatenate([img, img, img], axis=-1)
    return img[:, :, :2].astype(np.float32)


def make_measurement(rg: np.ndarray, K: np.ndarray) -> np.ndarray:
    """P = K * R^0.7 + (1-K) * G^2 + N(0, σ)"""
    R = rg[:, :, 0]
    G = rg[:, :, 1]
    P = K * R ** 0.7 + (1 - K) * G ** 2
    noise = np.random.normal(0, SIGMA, P.shape).astype(np.float32)
    return np.clip(P + noise, 0, 1)

In [8]:
def tv_loss(image: Tensor) -> Tensor:
    """
    Total Variation loss — возвращает скаляр.

    Работает для любого числа измерений (..., H, W).
    NaN-защита: clamp перед вычислением разностей.
    """
    
    image = image.clamp(-1e6, 1e6)

    diff_x = torch.abs(image[..., :, 1:] - image[..., :, :-1])
    diff_y = torch.abs(image[..., 1:, :] - image[..., :-1, :])

    tv = diff_x.mean() + diff_y.mean()

    # Если всё же получили NaN — вернуть ноль без обрыва графа
    if torch.isnan(tv):
        print(f"WARNING: TV is NaN! image range: [{image.min():.4f}, {image.max():.4f}]")
        return torch.zeros(1, device=image.device, requires_grad=True)[0]

    return tv   # скаляр, НЕ кортеж

In [10]:
def l0_grad_loss(image: Tensor, eps: float = 1e-3) -> Tensor:
    """
    Мягкая L0-аппроксимация нормы градиента:
        sum(|grad| / (|grad| + eps))
    При eps→0 стремится к L0 (счёт ненулевых градиентов).
    Поощряет кусочно-постоянные карты (piece-wise constant).
    """
    diff_x = torch.abs(image[..., :, 1:] - image[..., :, :-1])
    diff_y = torch.abs(image[..., 1:, :] - image[..., :-1, :])
    return (diff_x / (diff_x + eps)).mean() + (diff_y / (diff_y + eps)).mean()

In [11]:
def optimize(
    P,
    K,
    lambda_tv=0.02,
    lambda_order=0.1,
    lambda_l0=0.005,
    lr=0.001,
    max_iters=1000,
    tol=1e-8,
    grad_clip=1.0,
):

    U  = (P * (1.0 + 0.15 * (K - 0.5))).clone().detach().requires_grad_(True)
    Fi = (P * (1.0 - 0.15 * (K - 0.5))).clone().detach().requires_grad_(True)

    optimizer = torch.optim.Adam(
        [U, Fi], lr=lr, amsgrad=True
    )

    losses    = []
    prev_loss = float('inf')

    for i in tqdm(range(max_iters), desc="Optimization"):
        optimizer.zero_grad()

        #  Реконструкци
        P_recon = U * K + Fi * (1 - K)

        # Слагаемые лосса
        loss_l2    = F.mse_loss(P_recon, P)
        loss_tv_U  = tv_loss(U)    # скаляр (исправленный tv_loss)
        loss_tv_Fi = tv_loss(Fi)   # скаляр
        loss_order = (torch.relu(Fi - U) ** 2).mean()
        loss_l0    = l0_grad_loss(Fi)

        loss = (
            loss_l2
            + lambda_tv    * (loss_tv_U + loss_tv_Fi)
            + lambda_order * loss_order
            + lambda_l0    * loss_l0
        )

        loss.backward()

        # Gradient clipping — стабилизирует обучение
        torch.nn.utils.clip_grad_norm_([U, Fi], max_norm=grad_clip)

        optimizer.step()

        #  Ограничения 
        with torch.no_grad():
            U.clamp_(0, 20)
            Fi.clamp_(0, 2)
            Fi.copy_(torch.min(Fi, U * 0.99))  # Fi ≤ U·0.99 всегда

        losses.append(loss.item())

        if i % 100 == 0:
            tqdm.write(
                f"Iter {i:4d}: Loss={loss.item():.6f}  "
                f"L2={loss_l2.item():.6f}  "
                f"TV_U={loss_tv_U.item():.6f}  "
                f"TV_Fi={loss_tv_Fi.item():.6f}  "
                f"Order={loss_order.item():.6f}  "
                f"L0={loss_l0.item():.6f}"
            )

        #  Ранняя остановка
        delta = abs(prev_loss - loss.item())
        if delta < tol:
            tqdm.write(
                f"Converged at iter {i}: |Δloss|={delta:.2e} < tol={tol}"
            )
            break

        prev_loss = loss.item()

    return U.detach(), Fi.detach(), losses

In [12]:
# ЯЧЕЙКА 6: Предзагрузка изображений в RAM

def preload_images(paths, desc="Загрузка"):
    imgs = []
    for p in tqdm(paths, desc=desc):
        imgs.append(load_rg(p))
    t = torch.from_numpy(np.stack(imgs))
    t.share_memory_()
    return t


if CACHE_IMAGES:
    print("Предзагрузка train изображений...")
    train_images_tensor = preload_images(train_paths, desc="  train")
    print("Предзагрузка val изображений...")
    val_images_tensor   = preload_images(val_paths,   desc="  val  ")
    print(f"RAM: {(train_images_tensor.nbytes + val_images_tensor.nbytes) / 1e9:.2f} GB")
else:
    train_images_tensor = None
    val_images_tensor   = None

Предзагрузка train изображений...


  train: 100%|████████████████████████████████| 850/850 [03:26<00:00,  4.13it/s]


Предзагрузка val изображений...


  val  : 100%|████████████████████████████████| 150/150 [00:31<00:00,  4.76it/s]

RAM: 2.10 GB


In [13]:
# ЯЧЕЙКА 7: Dataset
# inp: (1, H, W) — только P; target: (2, H, W) — R и G

class DemuxDataset(Dataset):
    def __init__(
        self,
        images,
        configs,
        n_samples,
        patch_size=PATCH_SIZE,
        augment=True,
        seed=None,
    ):
        self.images     = images
        self.configs    = configs
        self.n_samples  = n_samples
        self.patch_size = patch_size
        self.augment    = augment

        n_img = images.shape[0] if isinstance(images, torch.Tensor) else len(images)
        H = images.shape[1] if isinstance(images, torch.Tensor) else 512
        W = images.shape[2] if isinstance(images, torch.Tensor) else 512

        if seed is not None:
            rng = np.random.default_rng(seed)
            self._img_idx = rng.integers(0, n_img,              n_samples)
            self._cfg_idx = rng.integers(0, len(configs),       n_samples)
            self._y0      = rng.integers(0, H - patch_size,     n_samples)
            self._x0      = rng.integers(0, W - patch_size,     n_samples)
        else:
            self._img_idx = None

    def _get_rg(self, img_idx):
        if isinstance(self.images, torch.Tensor):
            return self.images[img_idx].numpy().copy()
        return load_rg(self.images[img_idx])

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        ps = self.patch_size

        if self._img_idx is not None:
            img_idx = int(self._img_idx[idx])
            cfg_idx = int(self._cfg_idx[idx])
            y0      = int(self._y0[idx])
            x0      = int(self._x0[idx])
        else:
            n_img = (self.images.shape[0]
                     if isinstance(self.images, torch.Tensor)
                     else len(self.images))
            img_idx = np.random.randint(n_img)
            cfg_idx = np.random.randint(len(self.configs))
            H = self.images.shape[1] if isinstance(self.images, torch.Tensor) else 512
            W = self.images.shape[2] if isinstance(self.images, torch.Tensor) else 512
            y0 = np.random.randint(0, H - ps)
            x0 = np.random.randint(0, W - ps)

        rg     = self._get_rg(img_idx)
        config = self.configs[cfg_idx]
        H, W   = rg.shape[:2]

        K = make_mask(H, W, config)
        P = make_measurement(rg, K)

        P_p  = P [y0:y0+ps, x0:x0+ps].copy()
        rg_p = rg[y0:y0+ps, x0:x0+ps].copy()

        if self.augment:
            if np.random.rand() > 0.5:
                P_p  = np.fliplr(P_p).copy()
                rg_p = np.fliplr(rg_p).copy()
            if np.random.rand() > 0.5:
                P_p  = np.flipud(P_p).copy()
                rg_p = np.flipud(rg_p).copy()
            if np.random.rand() > 0.5:
                k90  = np.random.randint(1, 4)
                P_p  = np.rot90(P_p,  k90).copy()
                rg_p = np.rot90(rg_p, k90).copy()
            gamma = np.random.uniform(0.8, 1.2)
            rg_p  = np.clip(rg_p ** gamma, 0, 1)
            P_p   = np.clip(P_p  ** gamma, 0, 1)
            shift = np.random.uniform(-0.05, 0.05)
            rg_p  = np.clip(rg_p + shift, 0, 1)
            P_p   = np.clip(P_p  + shift, 0, 1)

        inp    = torch.tensor(P_p[None, :, :], dtype=torch.float32)
        target = torch.tensor(np.transpose(rg_p, (2, 0, 1)), dtype=torch.float32)
        return inp, target

In [14]:
def worker_init_fn(worker_id):
    np.random.seed(torch.initial_seed() % 2**32 + worker_id)


_images_train = train_images_tensor if CACHE_IMAGES else train_paths
_images_val   = val_images_tensor   if CACHE_IMAGES else val_paths

train_ds = DemuxDataset(
    _images_train, ALL_CONFIGS,
    n_samples=PATCHES_PER_EPOCH,
    augment=True, seed=None,
)
val_ds = DemuxDataset(
    _images_val, ALL_CONFIGS,
    n_samples=VAL_PATCHES,
    augment=False, seed=42,
)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True,
    worker_init_fn=worker_init_fn,
    prefetch_factor=4,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
    prefetch_factor=4,
)
print(f"Train батчей: {len(train_loader)} | Val батчей: {len(val_loader)}")


Train батчей: 1563 | Val батчей: 157


In [15]:
#  ЯЧЕЙКА 9: Модель — Deep UNetASPP (без FiLM)
#
# архитектуры:
#    • Энкодер расширен до 5 уровней:
#        enc1: 1   → 64
#        enc2: 64  → 128
#        enc3: 128 → 256
#        enc4: 256 → 512
#        enc5: 512 → 512    новый уровень
#    • ASPP боттлнек: 512 каналов, dilations [1,2,4,8,12],
#      double projection (512→256→out)
#    • Декодер симметричен энкодеру (5 шагов up-sampling)
#    • ~18M параметров вместо ~3M
# ──────────────────────────────────────────────────────────

class ConvBlock(nn.Module):
    """
    Два свёрточных слоя BN+ReLU с residual skip-connection.
    Dropout2d(p) между первым и вторым conv для регуляризации.
    """
    def __init__(self, in_ch, out_ch, drop=0.1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Dropout2d(drop),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
        # Residual: проекция 1×1 если размер каналов меняется
        self.skip = (
            nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, bias=False),
                          nn.BatchNorm2d(out_ch))
            if in_ch != out_ch else nn.Identity()
        )

    def forward(self, x):
        return self.block(x) + self.skip(x)   # residual



class ASPP(nn.Module):
    """
    Atrous Spatial Pyramid Pooling — расширенная версия.
    Dilations: [1, 2, 4, 8, 12] + global avg pool.
    Double projection для лучшего смешивания признаков.
    """
    def __init__(self, in_ch, out_ch):
        super().__init__()
        mid = out_ch // 4
        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(in_ch, mid, 3, padding=r, dilation=r, bias=False),
                nn.BatchNorm2d(mid),
                nn.ReLU(inplace=True),
            )
            for r in [1, 2, 4, 8, 12]   # ← было [1,2,4,8], добавлен 12
        ])
        self.gap = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_ch, mid, 1, bias=False),
            nn.ReLU(inplace=True),
        )
        # Double projection: (mid*6) → out_ch//2 → out_ch
        self.proj = nn.Sequential(
            nn.Conv2d(mid * 6, out_ch // 2, 1, bias=False),
            nn.BatchNorm2d(out_ch // 2),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch // 2, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Dropout2d(0.1),
        )

    def forward(self, x):
        h, w   = x.shape[2], x.shape[3]
        outs   = [b(x) for b in self.branches]
        gap_up = self.gap(x).expand(-1, -1, h, w)
        return self.proj(torch.cat(outs + [gap_up], dim=1))



class UNetASPP(nn.Module):

    def __init__(self, in_ch=1, out_ch=2):
        super().__init__()
        self.pool = nn.MaxPool2d(2)

        # ── Энкодер (5 уровней) ───────────────────────────
        self.enc1 = ConvBlock(in_ch,  64,  drop=0.05)
        self.enc2 = ConvBlock(64,    128, drop=0.1)
        self.enc3 = ConvBlock(128,   256, drop=0.1)
        self.enc4 = ConvBlock(256,   512, drop=0.15)
        self.enc5 = ConvBlock(512,   512, drop=0.15)  # ← новый

        # ── ASPP боттлнек ─────────────────────────────────
        self.bot  = ASPP(512, 512)

        # ── Декодер (5 шагов) ─────────────────────────────
        self.up5  = nn.ConvTranspose2d(512, 512, 2, stride=2)
        self.dec5 = ConvBlock(512 + 512, 256, drop=0.15)

        self.up4  = nn.ConvTranspose2d(256, 256, 2, stride=2)
        self.dec4 = ConvBlock(256 + 512, 128, drop=0.1)

        self.up3  = nn.ConvTranspose2d(128, 128, 2, stride=2)
        self.dec3 = ConvBlock(128 + 256, 64,  drop=0.1)

        self.up2  = nn.ConvTranspose2d(64,  64,  2, stride=2)
        self.dec2 = ConvBlock(64  + 128, 64,  drop=0.05)

        self.up1  = nn.ConvTranspose2d(64,  64,  2, stride=2)
        self.dec1 = ConvBlock(64  + 64,  64,  drop=0.05)

        self.out  = nn.Sequential(
            nn.Conv2d(64, out_ch, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        # Энкодер
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        e5 = self.enc5(self.pool(e4))        # ← новый уровень

        # Боттлнек ASPP
        b  = self.bot(self.pool(e5))

        # Декодер с skip-connections
        d5 = self.dec5(torch.cat([self.up5(b),  e5], dim=1))
        d4 = self.dec4(torch.cat([self.up4(d5), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))

        return self.out(d1)



device = torch.device("cuda:1")
model  = UNetASPP(in_ch=1, out_ch=2).to(device)
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model, device_ids=[1, 0], output_device=1)
    print("DataParallel: GPU 1 (главный) + GPU 0")

total_params = sum(p.numel() for p in model.parameters())
print(f"Устройство: {device}")
print(f"Параметров в модели: {total_params:,}")

DataParallel: GPU 1 (главный) + GPU 0
Устройство: cuda:1
Параметров в модели: 19,242,498


In [16]:
# ЯЧЕЙКА 10: Функция потерь — MSE + SSIM + Perceptual

class PerceptualLoss(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = models.vgg16(weights=models.VGG16_Weights.DEFAULT).features[:16].eval()
        for p in vgg.parameters():
            p.requires_grad = False
        self.vgg = vgg
        self.mse = nn.MSELoss()

    def forward(self, pred, target):
        p3 = pred[:,  :1].repeat(1, 3, 1, 1)
        t3 = target[:, :1].repeat(1, 3, 1, 1)
        return self.mse(self.vgg(p3), self.vgg(t3))


class CombinedLoss(nn.Module):
    def __init__(self, alpha=0.6, beta=0.2, gamma=0.2):
        super().__init__()
        self.alpha   = alpha
        self.beta    = beta
        self.gamma   = gamma
        self.mse     = nn.MSELoss()
        self.percept = PerceptualLoss().to(device)

    def ssim_loss(self, pred, target):
        mu_p  = F.avg_pool2d(pred,        11, stride=1, padding=5)
        mu_t  = F.avg_pool2d(target,      11, stride=1, padding=5)
        sp    = F.avg_pool2d(pred**2,     11, stride=1, padding=5) - mu_p**2
        st    = F.avg_pool2d(target**2,   11, stride=1, padding=5) - mu_t**2
        spt   = F.avg_pool2d(pred*target, 11, stride=1, padding=5) - mu_p*mu_t
        C1, C2 = 0.01**2, 0.03**2
        ssim_map = ((2*mu_p*mu_t + C1) * (2*spt + C2)) / \
                   ((mu_p**2 + mu_t**2 + C1) * (sp + st + C2))
        return 1 - ssim_map.mean()

    def forward(self, pred, target):
        l_mse  = self.mse(pred, target)
        l_ssim = self.ssim_loss(pred, target)
        l_perc = self.percept(pred, target)
        return self.alpha * l_mse + self.beta * l_ssim + self.gamma * l_perc


criterion = CombinedLoss(alpha=0.6, beta=0.2, gamma=0.2)

In [17]:
# ЯЧЕЙКА 11: Оптимизатор и scheduler

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=1e-4,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer,
    T_0=100,
    T_mult=2,
    eta_min=1e-6,
)

In [18]:
# ЯЧЕЙКА 12: Функции чекпоинтов

def save_checkpoint(path, epoch, model, optimizer, scheduler,
                    best_val_loss, patience_cnt,
                    train_history, val_history):
    torch.save({
        "epoch":           epoch,
        "model_state":     (model.module.state_dict()
                            if isinstance(model, nn.DataParallel)
                            else model.state_dict()),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "best_val_loss":   best_val_loss,
        "patience_cnt":    patience_cnt,
        "train_history":   train_history,
        "val_history":     val_history,
    }, path)
    print(f"  [ckpt] сохранён: {path}  (epoch {epoch})")


def load_checkpoint(path, model, optimizer, scheduler, scaler=None):
    ckpt = torch.load(path, map_location=device, weights_only=False)

    state  = ckpt["model_state"]
    target = model.module if isinstance(model, nn.DataParallel) else model

    first_key = next(iter(state.keys()))
    if not first_key.startswith("module."):
        target.load_state_dict(state)
    else:
        model.load_state_dict(state)

    optimizer.load_state_dict(ckpt["optimizer_state"])
    scheduler.load_state_dict(ckpt["scheduler_state"])

    if scaler is not None and "scaler_state" in ckpt:
        scaler.load_state_dict(ckpt["scaler_state"])
        print("  [ckpt] scaler восстановлен")

    print(f"  [ckpt] восстановлен: {path}  (epoch {ckpt['epoch']})")
    return (
        ckpt["epoch"] + 1,
        ckpt["best_val_loss"],
        ckpt["patience_cnt"],
        ckpt["train_history"],
        ckpt["val_history"],
    )


import sys

class Tee:
    def __init__(self, filename, mode="w"):
        self.file   = open(filename, mode)
        self.stdout = sys.stdout

    def write(self, data):
        self.file.write(data)
        self.stdout.write(data)

    def flush(self):
        self.file.flush()
        self.stdout.flush()

    def close(self):
        self.file.close()

tee = Tee("training_log(10).txt")

sys.stdout = tee
sys.stderr = tee

Чекпоинт не найден — обучение с нуля.
Epoch 000 [train]:   0%|          | 0/1563 [00:00<?, ?it/s]/home/user/.local/lib/python3.10/site-packages/torch/nn/modules/conv.py:456: UserWarning: Applied workaround for CuDNN issue, install nvrtc.so (Triggered internally at ../aten/src/ATen/native/cudnn/Conv_v8.cpp:80.)
  return F.conv2d(input, weight, bias, self.stride,
Epoch 000 [train]:   4%|3         | 59/1563 [00:51<20:29,  1.22it/s, loss=0.4707] 

In [ ]:
# ЯЧЕЙКА 13: Обучение
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

start_epoch   = 0
best_val_loss = float("inf")
patience_cnt  = 0
train_history = []
val_history   = []

scaler = torch.cuda.amp.GradScaler()

if os.path.exists(CHECKPOINT_PATH):
    print(f"Найден чекпоинт {CHECKPOINT_PATH} — возобновляем обучение...")
    start_epoch, best_val_loss, patience_cnt, train_history, val_history = \
        load_checkpoint(CHECKPOINT_PATH, model, optimizer, scheduler, scaler)
else:
    print("Чекпоинт не найден — обучение с нуля.")

for epoch in range(start_epoch, EPOCHS):

    # ── Train ──────────────────────────────────────────
    model.train()
    train_loss = 0.0
    loop = tqdm(train_loader, desc=f"Epoch {epoch:03d} [train]", leave=False)
    for inp, target in loop:
        inp, target = inp.to(device), target.to(device)
        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast():
            pred = model(inp)
            loss = criterion(pred, target)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()
        loop.set_postfix(loss=f"{loss.item():.4f}")

    # ── Validation ─────────────────────────────────────
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for inp, target in val_loader:
            inp, target = inp.to(device), target.to(device)
            with torch.amp.autocast('cuda'):
                val_loss += criterion(model(inp), target).item()

    train_loss /= len(train_loader)
    val_loss   /= len(val_loader)
    train_history.append(train_loss)
    val_history.append(val_loss)
    scheduler.step()

    if epoch % 10 == 0:
        epoch_model_path = os.path.join(ALL_MODELS_DIR, f"model_epoch_{epoch:03d}.pt")
        torch.save({
            "epoch":       epoch,
            "model_state": (model.module if isinstance(model, nn.DataParallel)
                            else model).state_dict(),
            "train_loss":  train_loss,
            "val_loss":    val_loss,
        }, epoch_model_path)
        print(f"  [all_models(3)] сохранена модель epoch {epoch:03d}")

    lr_now = optimizer.param_groups[0]["lr"]

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_cnt  = 0
        state = (model.module if isinstance(model, nn.DataParallel)
                 else model).state_dict()
        torch.save(state, BEST_MODEL_PATH)
        print(f"Epoch {epoch:03d}  train={train_loss:.4f}  val={val_loss:.4f}"
              f"  lr={lr_now:.6f}  ★ новый лучший")
    else:
        patience_cnt += 1
        if epoch % 25 == 0:
            print(f"Epoch {epoch:03d}  train={train_loss:.4f}  val={val_loss:.4f}"
                  f"  lr={lr_now:.6f}  patience={patience_cnt}/{PATIENCE}")
        if patience_cnt >= PATIENCE:
            print(f"\nEarly stopping на epoch {epoch}")
            torch.save({
                "epoch":           epoch,
                "model_state":     (model.module if isinstance(model, nn.DataParallel)
                                     else model).state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "scheduler_state": scheduler.state_dict(),
                "scaler_state":    scaler.state_dict(),
                "best_val_loss":   best_val_loss,
                "patience_cnt":    patience_cnt,
                "train_history":   train_history,
                "val_history":     val_history,
            }, CHECKPOINT_PATH)
            break

    if epoch % CHECKPOINT_EVERY == 0:
        torch.save({
            "epoch":           epoch,
            "model_state":     (model.module if isinstance(model, nn.DataParallel)
                                 else model).state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "scaler_state":    scaler.state_dict(),
            "best_val_loss":   best_val_loss,
            "patience_cnt":    patience_cnt,
            "train_history":   train_history,
            "val_history":     val_history,
        }, CHECKPOINT_PATH)
        print(f"  [ckpt] сохранён: {CHECKPOINT_PATH}  (epoch {epoch})")



In [ ]:
# Загружаем лучшие веса
state = torch.load(BEST_MODEL_PATH, map_location=device)
(model.module if isinstance(model, nn.DataParallel) else model).load_state_dict(state)
print(f"\nЛучший val_loss: {best_val_loss:.4f}")

# График потерь
plt.figure(figsize=(10, 4))
plt.plot(train_history, label="train")
plt.plot(val_history,   label="val")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Train vs Val loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# ЯЧЕЙКА 14: Визуализация результатов на val примерах

_infer_model = UNetASPP(in_ch=1, out_ch=2).to(device)
state = torch.load(BEST_MODEL_PATH, map_location=device)
_infer_model.load_state_dict(state)
_infer_model.eval()
print(f"Модель загружена: {BEST_MODEL_PATH}  |  val_loss best = {best_val_loss:.4f}")


@torch.no_grad()
def infer_patch(model, P: np.ndarray) -> np.ndarray:
    """P: (H,W) float32  →  rg_pred: (H,W,2) float32"""
    t = torch.tensor(P[None, None], dtype=torch.float32).to(device)
    with torch.amp.autocast('cuda'):
        out = model(t)
    return out[0].cpu().numpy().transpose(1, 2, 0)


def compute_metrics(pred_rg, gt_rg):
    from skimage.metrics import peak_signal_noise_ratio as psnr_fn
    from skimage.metrics import structural_similarity  as ssim_fn
    m = {}
    for i, ch in enumerate(["R", "G"]):
        m[f"PSNR_{ch}"] = psnr_fn(gt_rg[:,:,i], pred_rg[:,:,i], data_range=1.0)
        m[f"SSIM_{ch}"] = ssim_fn(gt_rg[:,:,i], pred_rg[:,:,i], data_range=1.0)
    m["PSNR_mean"] = (m["PSNR_R"] + m["PSNR_G"]) / 2
    m["SSIM_mean"] = (m["SSIM_R"] + m["SSIM_G"]) / 2
    return m


DEMO_CONFIGS = [
    ({"type": "h", "width": 16.0},                               "H-stripe 16px"),
    ({"type": "h", "width": 48.0},                               "H-stripe 48px"),
    ({"type": "s", "freq": 1/20, "angle": 0.0,  "phase": 0.0},  "Sin гориз T=20"),
    ({"type": "s", "freq": 1/15, "angle": 45.0, "phase": 1.0},  "Sin диаг  T=15"),
    ({"type": "s", "freq": 1/40, "angle": 90.0, "phase": 2.0},  "Sin верт  T=40"),
]

N_IMAGES = 4

fig_rows = N_IMAGES * len(DEMO_CONFIGS)
print(f"Всего примеров: {fig_rows}  ({N_IMAGES} изобр × {len(DEMO_CONFIGS)} масок)\n")

all_metrics = []

for img_i in range(N_IMAGES):
    rg = load_rg(val_paths[img_i])

    fig, axes = plt.subplots(
        len(DEMO_CONFIGS), 5,
        figsize=(18, 3.6 * len(DEMO_CONFIGS))
    )
    fig.suptitle(
        f"Val image #{img_i}  —  {val_paths[img_i].name if hasattr(val_paths[img_i], 'name') else img_i}",
        fontsize=12, y=1.01)

    for row, (config, label) in enumerate(DEMO_CONFIGS):
        K  = make_mask(512, 512, config)
        P  = make_measurement(rg, K)
        y0, x0 = 128, 128
        ps     = PATCH_SIZE
        P_p    = P [y0:y0+ps, x0:x0+ps]
        rg_p   = rg[y0:y0+ps, x0:x0+ps]
        K_p    = K [y0:y0+ps, x0:x0+ps]

        pred   = infer_patch(_infer_model, P_p)
        m      = compute_metrics(pred, rg_p)
        m["label"] = label
        all_metrics.append(m)

        panels = [
            (K_p,         "Маска K",  "gray",     0, 1),
            (P_p,         "Вход P",   "gray",     0, 1),
            (rg_p[:,:,0], "GT  R",    "Reds_r",   0, 1),
            (pred[:,:,0], "Pred R",   "Reds_r",   0, 1),
            (pred[:,:,1], "Pred G",   "Greens_r",0, 1),
        ]

        for col, (img, title, cmap, vmin, vmax) in enumerate(panels):
            ax = axes[row, col] if len(DEMO_CONFIGS) > 1 else axes[col]
            ax.imshow(img, cmap=cmap, vmin=vmin, vmax=vmax)
            if col == 0:
                ax.set_ylabel(label, fontsize=8, rotation=0,
                              labelpad=90, va="center")
            if row == 0:
                ax.set_title(title, fontsize=9)
            ax.axis("off")

        axes[row, -1].text(
            1.05, 0.5,
            f"PSNR R={m['PSNR_R']:.1f}\nPSNR G={m['PSNR_G']:.1f}\n"
            f"avg={m['PSNR_mean']:.1f} dB\nSSIM={m['SSIM_mean']:.3f}",
            transform=axes[row, -1].transAxes,
            fontsize=8, va="center",
            bbox=dict(boxstyle="round", fc="lightyellow", ec="gray", alpha=0.8),
        )

    plt.tight_layout()
    plt.savefig(f"val_image_{img_i}.png", dpi=110, bbox_inches="tight")
    plt.show()
    plt.close()

# Итоговая сводка
print("\n" + "="*62)
print(f"{'Конфигурация':<24}  {'PSNR_R':>6}  {'PSNR_G':>6}  {'PSNR avg':>8}  {'SSIM':>6}")
print("="*62)
for label in [c[1] for c in DEMO_CONFIGS]:
    rows = [m for m in all_metrics if m["label"] == label]
    print(f"  {label:<22}  "
          f"{np.mean([r['PSNR_R'] for r in rows]):>6.2f}  "
          f"{np.mean([r['PSNR_G'] for r in rows]):>6.2f}  "
          f"{np.mean([r['PSNR_mean'] for r in rows]):>8.2f}  "
          f"{np.mean([r['SSIM_mean'] for r in rows]):>6.4f}")
print("="*62)
print(f"  {'OVERALL':<22}  "
      f"{np.mean([m['PSNR_R'] for m in all_metrics]):>6.2f}  "
      f"{np.mean([m['PSNR_G'] for m in all_metrics]):>6.2f}  "
      f"{np.mean([m['PSNR_mean'] for m in all_metrics]):>8.2f}  "
      f"{np.mean([m['SSIM_mean'] for m in all_metrics]):>6.4f}")